# eFrog — Frog Call Classifier Training

Trains a multi-label CNN that identifies frog species from audio recordings, and exports it as
`frog_classifier.onnx` for the [efrog](https://github.com/lmansf/efrog) web app.

**Pipeline:** iNaturalist observations CSV → download audio → 16 kHz mel spectrograms → CNN → ONNX.

## Contract with the efrog server

`efrog/server.py` converts uploaded audio to a mel spectrogram and feeds it to this model, so the
constants below **must** stay in sync with the server:

| Constant | Value | Notes |
|---|---|---|
| `SAMPLE_RATE` | 16000 | mono |
| `DURATION` | 5.0 s | pad / truncate |
| `N_MELS` | 64 | |
| `N_FFT` | 1024 | |
| `HOP_LENGTH` | 512 | → 157 frames per 5 s clip |
| Input | `(batch, 1, 64, time)` float32 | `librosa` power mel → `power_to_db(ref=np.max)` |
| Output | `(batch, n_classes)` raw logits | the server applies sigmoid |

The class list is **embedded in the ONNX file's metadata** at export time, so the server picks up
the right labels automatically — no hand-maintained list to drift out of sync.

## How to run

```bash
pip install -r requirements.txt   # plus ffmpeg on your PATH (strongly recommended)
jupyter lab EDA-Master.ipynb      # Run All
```

Downloading the ~10k recordings takes a while (several GB). Downloads and feature extraction are
cached on disk, so re-running the notebook only does the missing work.

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# ── Audio / feature contract — MUST match efrog/server.py ──────────────────
SAMPLE_RATE  = 16000
DURATION     = 5.0
N_MELS       = 64
N_FFT        = 1024
HOP_LENGTH   = 512
CLIP_SAMPLES = int(SAMPLE_RATE * DURATION)
N_FRAMES     = 1 + CLIP_SAMPLES // HOP_LENGTH   # 157

# ── Dataset / training knobs ────────────────────────────────────────────────
MIN_RECORDINGS = 30     # drop species with fewer usable recordings
WINDOW_HOP_S   = 2.5    # stride between candidate 5 s windows
MAX_WINDOWS    = 4      # highest-energy windows kept per recording
VAL_FRACTION   = 0.15
BATCH_SIZE     = 64
EPOCHS         = 30
PATIENCE       = 6
LR             = 3e-4

DATA_DIR     = Path('data')
AUDIO_DIR    = DATA_DIR / 'audio'
FEATURE_DIR  = DATA_DIR / 'features'
ARTIFACT_DIR = Path('artifacts')
for d in (AUDIO_DIR, FEATURE_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} on {DEVICE} — expecting {N_FRAMES} mel frames per clip')

## 1. Load observations

iNaturalist research-grade observations of Florida frogs and toads that include a sound recording
(see `README.txt` for the export query and column descriptions).

In [ ]:
df = pd.read_csv('observations-master.csv')
df = df[df['sound_url'].notna() & (df['sound_url'] != '')].copy()
df['ext'] = df['sound_url'].str.split('?').str[0].str.rsplit('.', n=1).str[-1].str.lower()
print(f'{len(df)} observations with audio, {df.common_name.nunique()} species')
df[['id', 'common_name', 'scientific_name', 'sound_url', 'ext']].head()

## 2. Class distribution

The export contains a long tail of rare species. Species below `MIN_RECORDINGS` can't be learned
reliably, so they are dropped in the next step.

In [ ]:
counts = df['common_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [3, 1]})
counts.sort_values().plot.barh(ax=axes[0], color='seagreen')
axes[0].axvline(MIN_RECORDINGS, color='crimson', ls='--', label=f'min = {MIN_RECORDINGS}')
axes[0].set_title('Recordings per species')
axes[0].set_xlabel('recordings')
axes[0].legend()
df['ext'].value_counts().plot.bar(ax=axes[1], color='steelblue')
axes[1].set_title('Audio formats')
plt.tight_layout()
plt.show()

## 3. Select label classes

In [ ]:
keep    = counts[counts >= MIN_RECORDINGS]
dropped = counts[counts < MIN_RECORDINGS]

LABEL_CLASSES = sorted(keep.index)
df = df[df['common_name'].isin(LABEL_CLASSES)].copy()
df['label_idx'] = df['common_name'].map({c: i for i, c in enumerate(LABEL_CLASSES)})

print(f'{len(LABEL_CLASSES)} classes, {len(df)} recordings')
if len(dropped):
    print('dropped (too few recordings):', dict(dropped))
LABEL_CLASSES

## 4. Download audio

Files are cached as `data/audio/<observation id>.<ext>`. Failed downloads are reported at the end —
just re-run this cell to retry them.

In [ ]:
import urllib.request

def download_one(row) -> tuple[int, str | None]:
    dest = AUDIO_DIR / f'{row.id}.{row.ext}'
    if dest.exists() and dest.stat().st_size > 0:
        return row.id, None
    try:
        req = urllib.request.Request(row.sound_url, headers={'User-Agent': 'efrog-training/1.0'})
        with urllib.request.urlopen(req, timeout=30) as r, open(dest, 'wb') as f:
            shutil.copyfileobj(r, f)
        return row.id, None
    except Exception as exc:
        dest.unlink(missing_ok=True)
        return row.id, str(exc)

dl_errors = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(download_one, row) for row in df.itertuples()]
    for i, fut in enumerate(as_completed(futures), 1):
        rid, err = fut.result()
        if err:
            dl_errors[rid] = err
        if i % 1000 == 0 or i == len(futures):
            print(f'{i}/{len(futures)} processed ({len(dl_errors)} errors)')

df['audio_path'] = df.apply(lambda r: AUDIO_DIR / f'{r.id}.{r.ext}', axis=1)
df = df[df['audio_path'].map(lambda p: p.exists() and p.stat().st_size > 0)].copy()
print(f'{len(df)} recordings on disk; {len(dl_errors)} failed (re-run this cell to retry)')

## 5. Extract mel-spectrogram features

Each recording is decoded to 16 kHz mono (via ffmpeg when available — the same decode path the
server uses) and cut into 5 s windows. Frog calls are often sparse within a recording, so we keep
the `MAX_WINDOWS` highest-energy windows instead of random ones.

`clip_to_mel()` mirrors `audio_to_model_input()` in `efrog/server.py` exactly: power mel
spectrogram → `power_to_db(ref=np.max)` → `nan_to_num`. Features are cached as
`data/features/<id>.npy` with shape `(n_windows, 64, 157)`.

In [ ]:
HAVE_FFMPEG = shutil.which('ffmpeg') is not None
if not HAVE_FFMPEG:
    print('WARNING: ffmpeg not found — falling back to librosa decoding, '
          'which may fail on m4a/3gp files. Install ffmpeg for best results.')

def decode_audio(path: Path) -> np.ndarray:
    """Decode any audio format to 16 kHz mono float32 — same path as the efrog server."""
    if HAVE_FFMPEG:
        cmd = ['ffmpeg', '-y', '-i', str(path), '-ac', '1', '-ar', str(SAMPLE_RATE),
               '-f', 'f32le', '-loglevel', 'error', 'pipe:1']
        out = subprocess.run(cmd, capture_output=True, timeout=60)
        if out.returncode == 0 and out.stdout:
            return np.frombuffer(out.stdout, dtype=np.float32).copy()
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def clip_to_mel(clip: np.ndarray) -> np.ndarray:
    """Exactly mirrors audio_to_model_input() in efrog/server.py."""
    mel = librosa.feature.melspectrogram(
        y=clip, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return np.nan_to_num(mel_db, nan=0.0, posinf=0.0, neginf=-80.0).astype(np.float32)

def extract_windows(audio: np.ndarray) -> np.ndarray:
    """Up to MAX_WINDOWS highest-energy 5 s windows → (n, N_MELS, N_FRAMES)."""
    if len(audio) <= CLIP_SAMPLES:
        clips = [np.pad(audio, (0, CLIP_SAMPLES - len(audio)))]
    else:
        hop    = int(WINDOW_HOP_S * SAMPLE_RATE)
        cands  = [audio[s:s + CLIP_SAMPLES]
                  for s in range(0, len(audio) - CLIP_SAMPLES + 1, hop)]
        order  = np.argsort([-float(np.sqrt(np.mean(c ** 2))) for c in cands])
        clips  = [cands[i] for i in order[:MAX_WINDOWS]]
    return np.stack([clip_to_mel(c) for c in clips])

def extract_one(row) -> tuple[int, str | None]:
    dest = FEATURE_DIR / f'{row.id}.npy'
    if dest.exists():
        return row.id, None
    try:
        audio = decode_audio(row.audio_path)
        if len(audio) < SAMPLE_RATE // 2:    # < 0.5 s — unusable
            return row.id, 'too short'
        np.save(dest, extract_windows(audio))
        return row.id, None
    except Exception as exc:
        return row.id, str(exc)

feat_errors = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(extract_one, row) for row in df.itertuples()]
    for i, fut in enumerate(as_completed(futures), 1):
        rid, err = fut.result()
        if err:
            feat_errors[rid] = err
        if i % 1000 == 0 or i == len(futures):
            print(f'{i}/{len(futures)} extracted ({len(feat_errors)} errors)')

df = df[~df['id'].isin(feat_errors)].copy()
print(f'{len(df)} recordings with features')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, row in zip(axes.flat, df.groupby('common_name').head(1).itertuples()):
    ax.imshow(np.load(FEATURE_DIR / f'{row.id}.npy')[0],
              aspect='auto', origin='lower', cmap='magma')
    ax.set_title(row.common_name, fontsize=9)
fig.suptitle('Sample mel spectrograms (dB)')
plt.tight_layout()
plt.show()

## 6. Train / validation split and datasets

The split is **per recording** (stratified by species) so windows from one recording never appear
in both sets. Training uses SpecAugment-style frequency/time masking and a weighted sampler
(∝ 1/√class size) to soften the class imbalance.

In [ ]:
def grouped_split(frame: pd.DataFrame, val_fraction: float):
    val_ids = []
    for _, grp in frame.groupby('common_name'):
        ids = grp['id'].to_numpy().copy()
        rng.shuffle(ids)
        val_ids.extend(ids[:max(1, round(len(ids) * val_fraction))])
    val_mask = frame['id'].isin(val_ids)
    return frame[~val_mask].copy(), frame[val_mask].copy()

train_df, val_df = grouped_split(df, VAL_FRACTION)

def build_index(frame: pd.DataFrame):
    """One entry per (recording, window)."""
    items = []
    for row in frame.itertuples():
        n = np.load(FEATURE_DIR / f'{row.id}.npy', mmap_mode='r').shape[0]
        items.extend((row.id, w, row.label_idx) for w in range(n))
    return items

class FrogWindowDataset(Dataset):
    def __init__(self, items, n_classes, augment=False):
        self.items, self.n_classes, self.augment = items, n_classes, augment

    def __len__(self):
        return len(self.items)

    @staticmethod
    def _spec_augment(mel):
        floor = mel.min()
        for _ in range(2):   # frequency masks
            f0 = np.random.randint(0, N_MELS - 8)
            mel[f0:f0 + np.random.randint(1, 9), :] = floor
        for _ in range(2):   # time masks
            t0 = np.random.randint(0, mel.shape[1] - 20)
            mel[:, t0:t0 + np.random.randint(1, 21)] = floor
        return mel

    def __getitem__(self, i):
        rec_id, win, label = self.items[i]
        mel = np.load(FEATURE_DIR / f'{rec_id}.npy', mmap_mode='r')[win].copy()
        if self.augment:
            mel = self._spec_augment(mel)
        target = np.zeros(self.n_classes, dtype=np.float32)
        target[label] = 1.0
        return torch.from_numpy(mel).unsqueeze(0), torch.from_numpy(target)

train_items = build_index(train_df)
val_items   = build_index(val_df)
train_set   = FrogWindowDataset(train_items, len(LABEL_CLASSES), augment=True)
val_set     = FrogWindowDataset(val_items, len(LABEL_CLASSES))

class_n = np.bincount([lbl for _, _, lbl in train_items], minlength=len(LABEL_CLASSES))
weights = (1.0 / np.sqrt(np.maximum(class_n, 1)))[[lbl for _, _, lbl in train_items]]
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(train_items))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=2)
print(f'train: {len(train_df)} recordings / {len(train_set)} windows — '
      f'val: {len(val_df)} recordings / {len(val_set)} windows')

## 7. Model

A compact CNN (~1.5 M parameters → ~6 MB ONNX) keeps server cold-start and CPU inference fast.
Input normalization is baked into `forward()` so the server can feed raw dB spectrograms; the
output is one logit per class (multi-label — the server applies sigmoid per class, which lets
overlapping choruses score high for multiple species).

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        # GroupNorm rather than BatchNorm: identical behavior in train and eval
        # mode, so accuracy doesn't depend on batch statistics.
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(8, c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(8, c_out), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.net(x)

class FrogCNN(nn.Module):
    """Input: raw dB mel spectrogram (batch, 1, 64, time), values ≈ [-80, 0].
    Output: one logit per class.

    Pooling averages over time only — calls can occur anywhere in the clip, but
    pitch is a primary species cue, so the frequency layout is preserved."""
    def __init__(self, n_classes):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128), ConvBlock(128, 256),
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(256 * (N_MELS // 16), n_classes),
        )

    def forward(self, x):
        x = x / 40.0 + 1.0          # [-80, 0] dB → [-1, 1]
        x = self.blocks(x)          # (batch, 256, 4, time/16)
        x = x.mean(dim=3)           # pool over time only
        return self.head(x.flatten(1))

model = FrogCNN(len(LABEL_CLASSES)).to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters')

## 8. Train

In [ ]:
from sklearn.metrics import f1_score

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_eval(loader):
    model.eval()
    losses, preds, trues = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            losses.append(criterion(logits, y).item())
            preds.append(logits.argmax(1).cpu().numpy())
            trues.append(y.argmax(1).cpu().numpy())
    return float(np.mean(losses)), f1_score(
        np.concatenate(trues), np.concatenate(preds), average='macro')

best_f1, best_state, stale = -1.0, None, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(x)
    scheduler.step()

    val_loss, val_f1 = run_eval(val_loader)
    marker = ''
    if val_f1 > best_f1:
        best_f1   = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        stale, marker = 0, '  ← best'
    else:
        stale += 1
    print(f'epoch {epoch:2d}  train loss {epoch_loss / len(train_set):.4f}  '
          f'val loss {val_loss:.4f}  val macro-F1 {val_f1:.3f}{marker}')
    if stale >= PATIENCE:
        print(f'early stop — no improvement in {PATIENCE} epochs')
        break

model.load_state_dict(best_state)
print(f'best val macro-F1: {best_f1:.3f}')

## 9. Evaluate at recording level

The server classifies one 5 s clip, but the honest accuracy number is per recording: sigmoid
probabilities averaged over each validation recording's windows.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
rec_true, rec_pred = [], []
with torch.no_grad():
    for row in val_df.itertuples():
        mels  = np.load(FEATURE_DIR / f'{row.id}.npy')
        probs = torch.sigmoid(model(torch.from_numpy(mels).unsqueeze(1).to(DEVICE))).mean(0)
        rec_true.append(row.label_idx)
        rec_pred.append(int(probs.argmax()))

print(classification_report(rec_true, rec_pred,
                            labels=range(len(LABEL_CLASSES)),
                            target_names=LABEL_CLASSES, zero_division=0))

cm = confusion_matrix(rec_true, rec_pred, labels=range(len(LABEL_CLASSES)), normalize='true')
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(LABEL_CLASSES)), LABEL_CLASSES, rotation=90, fontsize=8)
ax.set_yticks(range(len(LABEL_CLASSES)), LABEL_CLASSES, fontsize=8)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('Recording-level confusion matrix (row-normalized)')
fig.colorbar(im)
plt.tight_layout()
plt.show()

## 10. Export to ONNX

Dynamic batch and time axes, input `input`, output `output` — the shape the server expects. The
label list and preprocessing constants are embedded as model metadata so the server reads them
straight from the file.

In [ ]:
import onnx

MODEL_PATH = ARTIFACT_DIR / 'frog_classifier.onnx'

model.eval().cpu()
torch.onnx.export(
    model, (torch.zeros(1, 1, N_MELS, N_FRAMES),), str(MODEL_PATH),
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size', 3: 'time'},
                  'output': {0: 'batch_size'}},
    opset_version=18,
    dynamo=False,
)

m = onnx.load(MODEL_PATH)
for key, value in {
    'labels':      json.dumps(LABEL_CLASSES),
    'sample_rate': str(SAMPLE_RATE),
    'duration':    str(DURATION),
    'n_mels':      str(N_MELS),
    'n_fft':       str(N_FFT),
    'hop_length':  str(HOP_LENGTH),
}.items():
    m.metadata_props.append(onnx.StringStringEntryProto(key=key, value=value))
onnx.checker.check_model(m)
onnx.save(m, MODEL_PATH)

(ARTIFACT_DIR / 'labels.json').write_text(json.dumps(LABEL_CLASSES, indent=2))
print(f'{MODEL_PATH} — {MODEL_PATH.stat().st_size / 1e6:.1f} MB, {len(LABEL_CLASSES)} classes')

## 11. Parity check against the efrog server

Re-implements the server's `audio_to_model_input()` on a validation file and confirms the exported
ONNX model agrees with the PyTorch model and carries the right labels.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
meta = sess.get_modelmeta().custom_metadata_map
assert json.loads(meta['labels']) == LABEL_CLASSES, 'label metadata mismatch'

def server_preprocess(path):
    """Re-implementation of audio_to_model_input() in efrog/server.py."""
    audio = decode_audio(path)
    if len(audio) < CLIP_SAMPLES:
        audio = np.pad(audio, (0, CLIP_SAMPLES - len(audio)))
    else:
        audio = audio[:CLIP_SAMPLES]
    return clip_to_mel(audio)[np.newaxis, np.newaxis]

sample = val_df.sample(1, random_state=SEED).iloc[0]
x = server_preprocess(sample.audio_path)

ort_logits   = sess.run(None, {'input': x})[0][0]
torch_logits = model(torch.from_numpy(x)).detach().numpy()[0]
assert np.allclose(ort_logits, torch_logits, atol=1e-4), 'ONNX and PyTorch disagree'

probs = 1 / (1 + np.exp(-ort_logits.astype(np.float64)))   # what the server does
print(f'true species: {sample.common_name}')
for i in np.argsort(probs)[::-1][:3]:
    print(f'  {probs[i]:.3f}  {LABEL_CLASSES[i]}')
print('\nONNX output matches PyTorch — model is ready for efrog.')

## 12. Deploy

Copy the model into the efrog repo (it replaces the old one) and redeploy:

```bash
cp artifacts/frog_classifier.onnx ../efrog/frog_classifier.onnx
```

The server reads the class list from the model's metadata, so no server code changes are needed
when the label set changes — `/health` will report the new classes.